# Aula 19 — MapReduce: N tarefas que nascem em runtime (última do curso)

**Objetivo:** paralelismo dinâmico — quando o N só se conhece executando.
Ao final você deve saber emitir `Send`, acumular com reducer e reduzir no final.

**Pré-requisitos:** Aula 08 (reducers), Aula 16 (fan-out) e Aula 07 (Pydantic).

**Como usar:** rode em ordem. Autocontido (1 arquivo no Colab). Comece pela
CÉLULA 0. Cole sua chave na CÉLULA 1 (só precisa no Colab/sem .env).
Com `GROQ_API_KEY` a avaliação é estruturada real; sem chave, tabela local.

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph langchain-groq python-dotenv
print('dependências prontas.')

In [ ]:
# CÉLULA 1 — chave do Groq (opcional: sem ela, tudo roda no Fake offline)
import getpass
import os
from dotenv import load_dotenv

load_dotenv()  # lê .env se existir (uso local); no Colab não há .env

if not os.getenv('GROQ_API_KEY'):
    try:
        os.environ['GROQ_API_KEY'] = getpass.getpass('GROQ_API_KEY (digitação oculta): ')
    except Exception:
        print('sem entrada interativa -> seguirei em modo Fake offline.')

print('chave presente:', bool(os.getenv('GROQ_API_KEY')))
# NUNCA dê print na chave. Verificamos só presença (True/False).

## 1. Analogia: a comanda com N itens vira N pratos

A mesa pede `pastel+refri+suco`. A cozinha **separa** em 3, cada prato sai **em
paralelo** com sua nota, e o garçom **elege o melhor** para sugerir da próxima vez.

| Restaurante | No LangGraph | O que é de verdade |
|---|---|---|
| separar a comanda | **nó `separa`** | divide o pedido em lista |
| um prato por item | **`Send('avalia', {...})`** | dispara 1 execução por item (N em runtime) |
| cada prato com sua nota | **sub-estado `Item`** | cada `Send` carrega só o seu |
| bandeja com tudo | **`avaliacoes: Annotated[list, add]`** | resultados somam sozinhos |
| eleger o melhor | **nó `elege`** (reduce final) | lê tudo e decide |

Diferença para a Aula 16: lá as arestas eram **fixas** (N conhecido no desenho);
aqui N nasce **executando** (pedido com 2 ou 20 itens, o desenho é o mesmo).

In [ ]:
# CÉLULA 2 — map dinâmico + reduce (rode com 3 e com 1 item)
from operator import add
from typing import Annotated, TypedDict
from langgraph.graph import START, END, StateGraph
from langgraph.types import Send   # novo (langgraph.constants.Send depreciado na v1.0)

class Comanda(TypedDict):
    pedido: str
    itens: list[str]
    avaliacoes: Annotated[list, add]   # o reduce do map: soma sozinho
    melhor: str

class Item(TypedDict):
    item: str

NOTAS = {'pastel': 9, 'refri': 7, 'suco': 8}

def separa(state: Comanda) -> dict:
    return {'itens': [i.strip() for i in state['pedido'].split('+') if i.strip()]}

def continua(state: Comanda):
    # MAP dinâmico: a condicional devolve LISTA de Send (um por item!)
    return [Send('avalia', {'item': i}) for i in state['itens']]

def avalia(state: Item) -> dict:
    return {'avaliacoes': [f"{state['item']}={NOTAS.get(state['item'], 5)}"]}

def elege(state: Comanda) -> dict:
    top = sorted(state['avaliacoes'], key=lambda s: int(s.split('=')[1]), reverse=True)[0]
    return {'melhor': top}

b = StateGraph(Comanda)
b.add_node('separa', separa)
b.add_node('avalia', avalia)
b.add_node('elege', elege)
b.add_edge(START, 'separa')
b.add_conditional_edges('separa', continua)
b.add_edge('avalia', 'elege')
b.add_edge('elege', END)
mapreduce = b.compile()

for pedido in ('pastel+refri+suco', 'pastel'):
    out = mapreduce.invoke({'pedido': pedido, 'itens': [], 'avaliacoes': [], 'melhor': ''})
    print(pedido, '->', out['avaliacoes'], '| melhor:', repr(out['melhor']))
assert mapreduce.invoke(
    {'pedido': 'pastel+refri+suco', 'itens': [], 'avaliacoes': [], 'melhor': ''})['melhor'] == 'pastel=9'
print('OK — mesmo desenho, N diferente: 3 avaliações aqui, 1 ali.')

## 2. Conceito: `Send` é aresta decidida em runtime

- Aresta fixa (`add_edge`): destino escrito no desenho, N conhecido.
- `Send('avalia', {...})`: **destino + sub-estado** decididos executando. A
  condicional devolve **lista** de `Send` — cada um vira 1 execução do nó.
- O sub-estado (`Item`) viaja com o `Send`: cada execução vê só o seu item
  (isolamento como sub-grafo da Aula 17, mas criado em loop).
- O reduce do map é o **reducer** (`Annotated[list, add]`): cada `avalia`
  anexa sem coordenação. O reduce **final** é um nó comum (`elege`) que roda
  quando todos terminam.
- Import novo: `from langgraph.types import Send` (`langgraph.constants.Send`
  do curso emite `LangGraphDeprecatedSinceV10`).

In [ ]:
# CÉLULA 3 — o stream prova o N dinâmico (rode e conte os 'avalia')
for pedido in ('pastel+refri+suco', 'pastel'):
    chaves = [list(c.keys()) for c in mapreduce.stream(
        {'pedido': pedido, 'itens': [], 'avaliacoes': [], 'melhor': ''})]
    print(pedido, '->', chaves)
    assert chaves.count(['avalia']) == len(pedido.split('+')), chaves
print('OK — 3 itens, 3 avalia; 1 item, 1 avalia. O desenho não mudou.')

## 3. Dissecando o `Send` (o que ele carrega)

- `Send('avalia', {'item': i})`: nome do nó + **sub-estado inicial** daquela
  execução. Chaves fora do schema do nó destino são ignoradas.
- A condicional `continua` **não devolve string** (como `decide_bebida` da Aula 02):
  devolve lista de `Send`. É outro 'modo' de aresta condicional.
- `avalia` roda N vezes **em paralelo** (Aula 16); cada uma escreve em
  `avaliacoes` via reducer — sem coordenação entre elas.
- `elege` espera todas (fan-in simétrico: mesma profundidade → 1x, Aula 16).

## 4. Variações que quebram (todas executadas e verificadas)

### 4a. Avaliação com saída estruturada (Groq real)

`with_structured_output(Pydantic)` dentro do map: cada avaliação sai tipada
(`item`, `nota: int`) em vez de string parseada. Com chave, validado aqui:
`pastel` avaliado com nota 0–10 pelo `gpt-oss-20b`.

In [ ]:
# CÉLULA 4 — structured output no map (Groq real com chave)
if os.getenv('GROQ_API_KEY'):
    from langchain_groq import ChatGroq
    from pydantic import BaseModel

    class Avaliacao(BaseModel):
        item: str
        nota: int

    llm = ChatGroq(model='openai/gpt-oss-20b', temperature=0, reasoning_effort='low')
    r = llm.with_structured_output(Avaliacao).invoke(
        "Avalie o pastel da lanchonete com nota de 0 a 10. Responda só o objeto.")
    print(type(r).__name__, r)
    assert isinstance(r.nota, int) and 0 <= r.nota <= 10, r
    print('OK — objeto tipado, sem parse de string.')
else:
    print('(sem chave: a tabela NOTAS do notebook faz o papel da avaliação;')
    print(' com a chave Groq, esta célula valida objeto tipado de verdade.)')

### 4b. `Send` com N=0 e N=1 (casos-limite)

- **N=0** (pedido vazio): nenhum `Send`, nenhuma avaliação — o `elege` roda
  com lista vazia. Seu reduce final precisa tolerar isso (aqui: `sorted([])`
  estouraria `IndexError` — trate!).
- **N=1**: 1 `Send`, 1 avaliação — map-reduce degenera para sequência, sem erro.

In [ ]:
# CÉLULA 5 — casos-limite: N=0 estoura, N=1 passa (rode e confira)
try:
    mapreduce.invoke({'pedido': '', 'itens': [], 'avaliacoes': [], 'melhor': ''})
    print('N=0 rodou?! (não deveria com este elege)')
except IndexError as erro:
    print('N=0 -> IndexError no elege (lista vazia, sem [0]): reduza com guarda!')

out1 = mapreduce.invoke({'pedido': 'suco', 'itens': [], 'avaliacoes': [], 'melhor': ''})
print('N=1:', out1)
assert out1['melhor'] == 'suco=8'
print('OK — N=1 degenera para sequência sem erro.')

## 5. Erros clássicos

1. **Desenhar N arestas para N dinâmico.** Se N nasce executando, aresta fixa
   não serve — é `Send` (CÉLULA 2). Aresta fixa é para N conhecido no desenho.
2. **Reduce final sem guarda para N=0.** Lista vazia + `[0]` = `IndexError`
   (CÉLULA 5). Todo `elege` precisa de fallback ('sem itens').
3. **Sub-estado fora do schema do nó.** Chave a mais no dict do `Send` é
   ignorada em silêncio — confira os nomes (`item` vs `itens`).
4. **Import antigo do `Send`.** `langgraph.constants.Send` funciona mas avisa
   depreciação (remoção na v2.0). Use `langgraph.types.Send`.
5. **Reduce sem reducer.** N `avalia` escrevendo na mesma etapa sem `add` =
   `InvalidUpdateError` (Aulas 08/16, agora com N dinâmico).
6. **Esperar ordem no map.** Avaliações chegam fora de ordem; `elege` ordena
   por nota, nunca por chegada (`sorting_reducer`, Aula 16, se precisar).

## 6. Exercício guiado — avaliador de 3 critérios

Situação: avaliar a **redação** em 3 critérios em paralelo (`clareza`,
`exemplos`, `conclusão`), cada um com nota fixa de mentira, e eleger a maior.
É o esqueleto do seu avaliador de redação em MapReduce.

In [ ]:
# CÉLULA 6 — exercício guiado (map de critérios + reduce)
from typing import Annotated as _A

class Redacao(TypedDict):
    texto: str
    criterios: list[str]
    notas: _A[list, add]
    melhor: str

class Criterio(TypedDict):
    criterio: str

NOTAS_CRIT = {'clareza': 8, 'exemplos': 9, 'conclusão': 7}

br2 = StateGraph(Redacao)
br2.add_node('separa', lambda s: {'criterios': ['clareza', 'exemplos', 'conclusão']})
br2.add_node('avalia_crit', lambda s: {'notas': [f"{s['criterio']}={NOTAS_CRIT[s['criterio']]}"]})
br2.add_node('elege', lambda s: {'melhor': sorted(
    s['notas'], key=lambda x: int(x.split('=')[1]), reverse=True)[0]})
br2.add_edge(START, 'separa')
br2.add_conditional_edges('separa', lambda s: [Send('avalia_crit', {'criterio': c}) for c in s['criterios']])
br2.add_edge('avalia_crit', 'elege')
br2.add_edge('elege', END)
outr = br2.compile().invoke({'texto': 'minha redação', 'criterios': [], 'notas': [], 'melhor': ''})
print(outr['notas'], '| melhor:', repr(outr['melhor']))
assert outr['melhor'] == 'exemplos=9', outr
print('OK — 3 critérios em paralelo, melhor elegido. Troque NOTAS_CRIT por LLM depois.')

## 7. Desafio — MapReduce com LLM de verdade nos 2 níveis

Com a chave Groq, evolua o exercício:

1. `separa` usa o LLM com saída estruturada (`criterios: list[str]`) a partir
   do **texto** (não mais lista fixa).
2. `avalia_crit` usa o LLM com saída estruturada (`nota: int`) por critério.
3. `elege` monta a média ponderada (pesos: clareza 40%, exemplos 30%,
   conclusão 30%) em vez do máximo.

- Valide com 2 redações (uma boa, uma fraca): a boa pontua mais.
- **Pegadinha:** N notas vindas do LLM podem empatar ou vir fora de 0–10 —
  valide com Pydantic (Aula 07) antes de ponderar.

## 8. Checklist + FECHAMENTO DO CURSO (00–19)

**Você entendeu se consegue responder sem olhar:**
- [ ] Por que aresta fixa não serve quando N nasce executando?
- [ ] O que um `Send` carrega — e o que acontece com chave fora do schema?
- [ ] Quem soma os resultados do map (e o que falta sem ele)?
- [ ] Reduce final com N=0: o que quebra e como proteger?
- [ ] `langgraph.constants.Send` vs `langgraph.types.Send`: qual usar?
- [ ] No seu avaliador, o que seria map (critérios?) e o que seria reduce (nota final?)?

**O curso em uma frase por bloco:**
- 00–02: grafo = estado + nós + arestas; `compile`/`invoke`; overwrite.
- 03–05: mensagens, cardápio que pede (`bind_tools`), caixa que executa
  (`ToolNode`) e a volta que cria o agente ReAct.
- 06: sessão (`checkpointer` + `thread_id`) vira invokes em conversa.
- 07–09: contratos (`TypedDict`/Pydantic), mesclagem (reducers) e API enxuta
  (privado + `input_schema`/`output_schema`).
- 10–11: poda do histórico (`trim`/`filter`) e ver executando (`stream`).
- 12–15: humano-no-loop (pausa, edição, régua, replay e fork no tempo).
- 16–19: escala (fan-out/in, sub-grafos, observabilidade, MapReduce).

**E agora?** O próximo passo natural é aplicar tudo no **seu avaliador de
redação com LangGraph**: rubrica como schema, critérios em MapReduce,
memória por aluno e pausa para revisão humana. É só chamar.